In [35]:
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

from sklearn.metrics import root_mean_squared_error

import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

from src.features.feature_utils import *

In [2]:
raw_train_loc = "../data/raw/train.xlsx"
raw_test_loc = "../data/raw/test.xlsx"

raw_train = pd.read_excel(raw_train_loc)
raw_test = pd.read_excel(raw_test_loc)

In [4]:
raw_train_lag_week = raw_train[-168:]

In [5]:
raw_test_with_lag_week = pd.concat([raw_train_lag_week, raw_test], axis = 0)

test_transformed = complete_df_transformer(raw_test_with_lag_week)

In [11]:
test_transformed.head()

,Year,Month,Day,Hour,Load,Site-1 Temp,Site-2 Temp,Site-3 Temp,Site-4 Temp,Site-5 Temp,...,Load_lag_1h,Load_lag_2h,Load_lag_3h,Load_lag_6h,Load_lag_12h,Load_lag_24h,Load_lag_48h,Load_lag_168h,is_weekend,is_notable_day
0,2022,1,1,0,2064,54.14,52.70,48.74,54.86,51.44,...,2118.0,2229.0,2348.0,2644.0,2184.0,1968.0,2203.0,2008.0,1,1
1,2022,1,1,1,1975,53.78,51.98,48.74,54.14,51.08,...,2064.0,2118.0,2229.0,2545.0,2193.0,1916.0,2125.0,1887.0,1,1
2,2022,1,1,2,1900,51.44,50.36,48.20,53.24,49.82,...,1975.0,2064.0,2118.0,2449.0,2280.0,1851.0,2115.0,1814.0,1,1
3,2022,1,1,3,1829,50.00,50.54,48.74,51.62,48.02,...,1900.0,1975.0,2064.0,2348.0,2359.0,1799.0,2092.0,1776.0,1,1
4,2022,1,1,4,1828,51.98,50.54,49.64,51.62,49.28,...,1829.0,1900.0,1975.0,2229.0,2513.0,1856.0,1997.0,1776.0,1,1


In [13]:
with open('../trained_models/baseline_linear.pkl', 'rb') as file:
    baseline_linear = pickle.load(file)

with open('../trained_models/engineered_linear.pkl', 'rb') as file:
    engineered_linear = pickle.load(file)

with open('../trained_models/complete_xgb.pkl', 'rb') as file:
    complete_xgb = pickle.load(file)

with open('../trained_models/lin_xgb_linear_component.pkl', 'rb') as file:
    lin_xgb_linear_component = pickle.load(file)

with open('../trained_models/lin_xgb_xgboost_component.pkl', 'rb') as file:
    lin_xgb_xgboost_component = pickle.load(file)


In [15]:
def generate_all_predictions(model_name, df):
    
    match model_name:
        case "baseline_linear":
            predict_df = df[["Month_sin", "Month_cos", "Day", "Hour_sin", "Hour_cos", "avg_region_temp", "avg_region_ghi"]]
            return baseline_linear.predict(predict_df)
            
        case "engineered_linear":
            predict_df = df[["Hour_sin", "Hour_cos", "Month", "Day", "temp_6h", "avg_region_ghi", "is_weekend", "is_notable_day", "CDH", "HDH", "Load_lag_1h", "Load_lag_2h", "Load_lag_3h", "Load_lag_24h"]]
            return engineered_linear.predict(predict_df)
            
        case "complete_xgb":
            predict_df = df.drop(columns = ["Load", "timestamp"])
            return complete_xgb.predict(predict_df)
            
        case "lin_xgb":
            predict_df = df.drop(columns = ["Load", "timestamp"])

            lin_features = ["Hour_sin", "Hour_cos", "Month", "Day", "temp_6h", "avg_region_ghi", "is_weekend", "is_notable_day", "CDH", "HDH", "Load_lag_1h", "Load_lag_2h", "Load_lag_3h", "Load_lag_24h"]
            
            linear = lin_xgb_linear_component.predict(predict_df[lin_features])
            
            residual = lin_xgb_xgboost_component.predict(predict_df)
            
            return linear + residual

        case _:
            raise ValueError(f"Invalid model name: {model_name}")


In [17]:
baseline_preds = pd.DataFrame(generate_all_predictions("baseline_linear", baseline_df_transformer(raw_test)), columns = ["baseline linear"])

eng_lin_preds = pd.DataFrame(generate_all_predictions("engineered_linear", test_transformed), columns = ["engineered linear"])

xgb_preds = pd.DataFrame(generate_all_predictions("complete_xgb", test_transformed), columns = ["complete xgboost"])

lin_xgb_preds = pd.DataFrame(generate_all_predictions("lin_xgb", test_transformed), columns = ["linear and xgboost"])



In [45]:
complete_predictions = pd.concat([test_transformed, baseline_preds, eng_lin_preds, xgb_preds, lin_xgb_preds], axis = 1)
complete_predictions = complete_predictions[["timestamp", "Load", "baseline linear", "engineered linear", "complete xgboost", "linear and xgboost", "avg_region_temp", "is_weekend"]].rename(columns = {"Load": "Actual Load"})
complete_predictions.head()



,timestamp,Actual Load,baseline linear,engineered linear,complete xgboost,linear and xgboost,avg_region_temp,is_weekend
0,2022-01-01 00:00:00,2064,1853.168547,1969.306256,1994.155396,2012.380732,52.376,1
1,2022-01-01 01:00:00,1975,1781.970529,1969.865069,1940.527466,1987.329755,51.944,1
2,2022-01-01 02:00:00,1900,1705.689739,1853.800061,1876.505859,1869.763754,50.612,1
3,2022-01-01 03:00:00,1829,1669.708819,1814.433098,1836.733643,1827.542296,49.784,1
4,2022-01-01 04:00:00,1828,1710.003934,1757.616487,1803.551758,1788.300978,50.612,1


In [53]:
print(root_mean_squared_error(complete_predictions["Actual Load"], complete_predictions["baseline linear"]))

print(root_mean_squared_error(complete_predictions["Actual Load"], complete_predictions["engineered linear"]))

print(root_mean_squared_error(complete_predictions["Actual Load"], complete_predictions["complete xgboost"]))

print(root_mean_squared_error(complete_predictions["Actual Load"], complete_predictions["linear and xgboost"]))


332.20271973675216
68.15219342160108
68.8816764322244
56.86686013814809


In [65]:
top_5_percent_load_val = complete_predictions["Actual Load"].quantile(0.95) 
top_5_percent_load_val_df = complete_predictions[complete_predictions["Actual Load"] >= top_5_percent_load_val]


print(root_mean_squared_error(top_5_percent_load_val_df["Actual Load"], top_5_percent_load_val_df["baseline linear"]))

print(root_mean_squared_error(top_5_percent_load_val_df["Actual Load"], top_5_percent_load_val_df["engineered linear"]))

print(root_mean_squared_error(top_5_percent_load_val_df["Actual Load"], top_5_percent_load_val_df["complete xgboost"]))

print(root_mean_squared_error(top_5_percent_load_val_df["Actual Load"], top_5_percent_load_val_df["linear and xgboost"]))



841.5747375836273
86.75823194835975
135.90860786851957
74.4069369900446


In [69]:
top_1_percent_load_val = complete_predictions["Actual Load"].quantile(0.99) 
top_1_percent_load_val_df = complete_predictions[complete_predictions["Actual Load"] >= top_1_percent_load_val]


print(root_mean_squared_error(top_1_percent_load_val_df["Actual Load"], top_1_percent_load_val_df["baseline linear"]))

print(root_mean_squared_error(top_1_percent_load_val_df["Actual Load"], top_1_percent_load_val_df["engineered linear"]))

print(root_mean_squared_error(top_1_percent_load_val_df["Actual Load"], top_1_percent_load_val_df["complete xgboost"]))

print(root_mean_squared_error(top_1_percent_load_val_df["Actual Load"], top_1_percent_load_val_df["linear and xgboost"]))



1184.2506964726656
107.4316034561639
236.25642158303555
106.24516162012387


In [23]:
test_concat.columns

Index(['Year', 'Month', 'Day', 'Hour', 'Load', 'Site-1 Temp', 'Site-2 Temp',
       'Site-3 Temp', 'Site-4 Temp', 'Site-5 Temp', 'Site-1 GHI', 'Site-2 GHI',
       'Site-3 GHI', 'Site-4 GHI', 'Site-5 GHI', 'avg_region_temp',
       'avg_region_ghi', 'timestamp', 'Hour_sin', 'Hour_cos', 'temp_3h',
       'temp_6h', 'temp_24h', 'CDH', 'HDH', 'CDH_3h', 'CDH_6h', 'CDH_24h',
       'HDH_3h', 'HDH_6h', 'HDH_24h', 'Load_lag_1h', 'Load_lag_2h',
       'Load_lag_3h', 'Load_lag_6h', 'Load_lag_12h', 'Load_lag_24h',
       'Load_lag_48h', 'Load_lag_168h', 'is_weekend', 'is_notable_day',
       'baseline linear', 'engineered linear', 'complete xgboost',
       'linear and xgboost'],
      dtype='object')

In [54]:
test_df=raw_train[0:5].copy()

In [58]:
outs = generate_prediction(test_df, "baseline_linear")

In [60]:
print(outs)
print(test_df["Load"])

[1659.28934084 1588.09132313 1542.65495268 1505.57244627 1518.3279011 ]
0    1997
1    1921
2    1861
3    1833
4    1847
Name: Load, dtype: int64
